In [ ]:
from google.colab import drive
drive.mount("/content/gdrive")

In [ ]:
!rm -r rag_data_extracted

In [ ]:
!unzip rag_data.zip

In [ ]:
!cp rag_data/colab_rag_basic.py .

In [ ]:
!pip install -q langchain langchain-community langchain-text-splitters chromadb sentence-transformers

In [ ]:
# ==============================================================================
# Google Colab'de çalıştırılacak Temel RAG (Basic RAG) Kodu
# ==============================================================================
# Bu dosyayı doğrudan Google Colab hücresine kopyalayıp çalıştırabilirsiniz.
# Çalıştırmadan önce sol menüden 'rag_data.zip' dosyasını yüklemeyi unutmayın!


import os
import re
import zipfile
import shutil
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

# 2. ADIM: Yüklediğiniz rag_data.zip dosyasını açıyoruz
zip_path = "rag_data.zip"
extract_dir = "rag_data_extracted"

if not os.path.exists(zip_path):
    print(f"🔴 HATA: Lütfen sol taraftaki dosya menüsünden '{zip_path}' dosyasını Colab'e yükleyin!")
else:
    if os.path.exists(extract_dir):
        shutil.rmtree(extract_dir)

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print("[✓] ZIP dosyası başarıyla açıldı!")

    # 3. ADIM: Nomic Embed (v1.5) modelini yüklüyoruz (Colab GPU'su aktif ise CUDA'da çalışır)
    print("[*] Nomic Embed modeli HuggingFace üzerinden yükleniyor (Lütfen bekleyin)...")
    embeddings = HuggingFaceEmbeddings(
        model_name="nomic-ai/nomic-embed-text-v1.5",
        model_kwargs={"trust_remote_code": True, "device": "cuda" if os.system("nvidia-smi > /dev/null 2>&1") == 0 else "cpu"}
    )
    print("[✓] Embedding modeli yüklendi.")

    # 4. ADIM: Parçalayıcıları Tanımlıyoruz (İki Aşamalı Chunking)
    # Makale/Tez süreçlerinde raporlamak için parametreleri tanımlıyoruz
    CHUNK_SIZE = 800
    CHUNK_OVERLAP = 150

    # 1. Aşama: Başlıkları yakalamak için Markdown parçalayıcı
    headers_to_split_on = [
        ("#", "header_1"),
        ("##", "header_2"),
        ("###", "header_3"),
        ("####", "header_4"),
    ]
    markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

    # 2. Aşama: Büyük bölümleri alt parçalara bölmek için karakter sınırlı parçalayıcı
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP
    )

    all_chunks = []

    # 5. ADIM: NIDDK Dosyasını Parçalıyoruz (Sayfa Sayfa Ayrıştırarak)
    niddk_file_path = ""
    # ZIP içindeki dizin yapısına göre dinamik olarak dosyayı buluyoruz
    for root, dirs, files in os.walk(extract_dir):
        if "NIDDK_Guiding-Principles_clean.md" in files:
            niddk_file_path = os.path.join(root, "NIDDK_Guiding-Principles_clean.md")
            break

    if niddk_file_path and os.path.exists(niddk_file_path):
        print(f"[*] NIDDK belgesi işleniyor: {os.path.basename(niddk_file_path)}")
        with open(niddk_file_path, "r", encoding="utf-8") as f:
            niddk_content = f.read()

        # Sayfa yorumlarına göre bölüyoruz <!-- PAGE: X -->
        parts = re.split(r'<!-- PAGE:\s*(\d+)\s*-->', niddk_content)

        pages = []
        # İlk etiket öncesi önsöz metni varsa sayfa 7 (preface) olarak atıyoruz
        if parts[0].strip():
            pages.append((7, parts[0]))

        for i in range(1, len(parts), 2):
            page_no = int(parts[i])
            page_text = parts[i+1]
            pages.append((page_no, page_text))

        # Her sayfayı kendi içinde parçalıyoruz
        for page_no, page_text in pages:
            if not page_text.strip():
                continue
            # 1. Aşama: Başlıkları çıkar
            header_docs = markdown_splitter.split_text(page_text)
            # 2. Aşama: Metni limitlere böl ve metadatayı koru
            split_docs = text_splitter.split_documents(header_docs)

            for doc in split_docs:
                # Metaverileri zenginleştiriyoruz
                doc.metadata["source"] = "NIDDK_Guiding-Principles_clean.md"
                doc.metadata["guideline"] = "NIDDK"
                doc.metadata["page"] = page_no

                # Nomic için document öneki ekleme (Önerilen yöntem)
                doc.page_content = f"search_document: {doc.page_content}"
                all_chunks.append(doc)
    else:
        print("🔴 NIDDK belgesi bulunamadı!")

    # 6. ADIM: ADA Dosyalarını Parçalıyoruz (Bölüm Bölüm, Sayfasız)
    ada_dir_path = ""
    for root, dirs, files in os.walk(extract_dir):
        if "ADA" in root and any(f.endswith(".md") for f in files):
            ada_dir_path = root
            break

    if ada_dir_path and os.path.exists(ada_dir_path):
        print(f"[*] ADA belgeleri işleniyor (Dizin: {ada_dir_path})...")
        ada_files = [f for f in os.listdir(ada_dir_path) if f.endswith(".md")]

        for ada_file in ada_files:
            file_path = os.path.join(ada_dir_path, ada_file)
            with open(file_path, "r", encoding="utf-8") as f:
                ada_content = f.read()

            # 1. Aşama: Başlıkları çıkar
            header_docs = markdown_splitter.split_text(ada_content)
            # 2. Aşama: Alt parçalara böl
            split_docs = text_splitter.split_documents(header_docs)

            for doc in split_docs:
                doc.metadata["source"] = ada_file
                doc.metadata["guideline"] = "ADA"
                doc.metadata["page"] = -1  # Sayfa olmadığı için sentinel değer atıyoruz

                # Nomic için document öneki
                doc.page_content = f"search_document: {doc.page_content}"
                all_chunks.append(doc)
    else:
        print("🔴 ADA belgeleri klasörü bulunamadı!")

    # NIDDK ve ADA parçalarının sayılarını hesaplıyoruz
    niddk_chunk_count = sum(1 for doc in all_chunks if doc.metadata["guideline"] == "NIDDK")
    ada_chunk_count = sum(1 for doc in all_chunks if doc.metadata["guideline"] == "ADA")
    total_chunks = len(all_chunks)

    # Akademik makale/tez raporlaması için istatistik tablosunu yazdırıyoruz
    print("\n" + "="*60)
    print("           RAG BELGE PARÇALAMA & PARAMETRE İSTATİSTİKLERİ")
    print("="*60)
    print(f"[*] Deneysel Kurulum (Experimental Setup):")
    print(f"    - Chunk Size (Parça Sınırı): {CHUNK_SIZE} karakter")
    print(f"    - Chunk Overlap (Üst Üste Binme): {CHUNK_OVERLAP} karakter")
    print(f"    - Embedding Modeli (Model Name): nomic-ai/nomic-embed-text-v1.5")
    print(f"    - Embedding Boyutu (Dimension): 768")
    print("-"*60)
    print(f"[*] Dağılım ve Parça Sayıları (Dataset Statistics):")
    print(f"    - NIDDK Rehberi Chunks: {niddk_chunk_count} adet (%{niddk_chunk_count / total_chunks * 100:.2f})")
    print(f"    - ADA Rehberi Chunks: {ada_chunk_count} adet (%{ada_chunk_count / total_chunks * 100:.2f})")
    print(f"    - Toplam Parça Sayısı (Total Chunks): {total_chunks} adet")
    print("="*60 + "\n")

    # 7. ADIM: Chroma Vektör Veri Tabanını oluşturuyoruz
    db_dir = "chroma_db"

    # SQLite kilitlenmelerini önlemek için Chroma sistem önbelleğini temizliyoruz (Readonly DB hatasını çözer)
    try:
        import chromadb
        chromadb.api.client.SharedSystemClient.clear_system_cache()
    except Exception:
        pass

    if os.path.exists(db_dir):
        shutil.rmtree(db_dir)

    print("[*] Chroma veri tabanı oluşturuluyor ve embeddingler hesaplanıyor...")
    db = Chroma.from_documents(all_chunks, embeddings, persist_directory=db_dir)
    print(f"[✓] Chroma veri tabanı başarıyla oluşturuldu ve '{db_dir}' klasörüne kaydedildi.")

    # 8. ADIM: TEST SORGUSU (Temel RAG Araması)
    query = "What is the diagnosis criteria for diabetes?"
    # Nomic sorguları için search_query: ön eki ekliyoruz
    nomic_query = f"search_query: {query}"

    print(f"\n🔎 TEST SORGUSU: '{query}'")
    results = db.similarity_search(nomic_query, k=3)

    for idx, doc in enumerate(results, 1):
        # Nomic ön ekini temizleyerek yazdırıyoruz
        content_clean = doc.page_content.replace("search_document: ", "", 1)
        print(f"\n[{idx}] Skor/Eşleşme:")
        print(f"    - Kaynak: {doc.metadata.get('source')}")
        print(f"    - Kılavuz: {doc.metadata.get('guideline')}")
        print(f"    - Sayfa: {doc.metadata.get('page')}")
        print(f"    - Başlıklar: {doc.metadata.get('header_1')} > {doc.metadata.get('header_2', '')}")
        print(f"    - İçerik: {content_clean[:200]}...")


In [ ]:

    # 8. ADIM: TEST SORGUSU (Temel RAG Araması)
    query = "What is the diagnosis criteria for diabetes?"
    # Nomic sorguları için search_query: ön eki ekliyoruz
    nomic_query = f"search_query: {query}"

    print(f"\n🔎 TEST SORGUSU: '{query}'")
    results = db.similarity_search(nomic_query, k=3)

    for idx, doc in enumerate(results, 1):
        # Nomic ön ekini temizleyerek yazdırıyoruz
        content_clean = doc.page_content.replace("search_document: ", "", 1)
        print(f"\n[{idx}] Skor/Eşleşme:")
        print(f"    - Kaynak: {doc.metadata.get('source')}")
        print(f"    - Kılavuz: {doc.metadata.get('guideline')}")
        print(f"    - Sayfa: {doc.metadata.get('page')}")
        print(f"    - Başlıklar: {doc.metadata.get('header_1')} > {doc.metadata.get('header_2', '')}")
        print(f"    - İçerik: {content_clean[:]}...")
